# Bronze: Delta (ADLS Gen2) -> `edw` dedicated SQL pool (`brz` schema)

Incrementally loads every Delta table found under the configured ADLS roots into
the `edw` dedicated SQL pool, using:

- **Delta Change Data Feed (CDF)** for change detection (open-source Delta; no Databricks).
- **Append-only** writes (bronze keeps the full change log; dbt de-dupes later).
- **Workspace managed identity** auth only (no keys/passwords).
- **Uniform physical design**: every `brz` table is `ROUND_ROBIN` + `CLUSTERED COLUMNSTORE`.
- **All business columns as `NVARCHAR`** (no type inference); operational metadata columns stay typed for correct idempotency.
- **Idempotent, exactly-once per run** via an overwrite-staging table + a single transactional swap, with the watermark advanced only on commit.
- **Per-table failure isolation** + thorough logging to stdout and to `edw.operations`.

Run it standalone (defaults below) for testing, or from a Synapse pipeline Notebook activity (override the Parameters cell).

Prerequisite: run `sql/00_setup.sql` once, and enable CDF on source tables.

In [ ]:
# === Parameters ===
# In Synapse, toggle this cell as the 'Parameters' cell so a pipeline can override it.
# All values have safe defaults so the notebook also runs standalone for testing.

sql_endpoint   = ""          # REQUIRED, e.g. "myws.sql.azuresynapse.net"
database       = "edw"
target_schema  = "brz"
ops_schema     = "operations"

# One or more ADLS Gen2 roots to scan for Delta tables.
# Comma-separated string (from a pipeline) or a Python list both work.
source_paths   = ""          # REQUIRED, e.g. "abfss://raw@acct.dfs.core.windows.net/delta"

# Staging folder used by the dedicated-pool connector for COPY.
# The workspace managed identity needs Storage Blob Data Contributor here.
staging_path   = ""          # REQUIRED, e.g. "abfss://staging@acct.dfs.core.windows.net/synapse_tmp"

string_len     = 4000        # default NVARCHAR length for every business column
max_workers    = 4           # parallel tables (respect SQL pool concurrency slots)
token_audience = "DW"        # AAD audience for the dedicated SQL pool token
config_path    = ""          # optional abfss path to config/tables.json
fail_on_error  = False       # if True, raise at end when any table failed

# Restrict which discovered tables to process. Names are the bronze table names
# (= sanitized source folder names). Comma-separated string or Python list.
# Leave both blank to process everything that is discovered.
include_tables = ""          # process ONLY these, e.g. "customers,orders"
exclude_tables = ""          # skip these, e.g. "tmp_scratch"

In [ ]:
# === Imports, logging, run context, optional config ===
import json, re, uuid, traceback, datetime, concurrent.futures
import logging

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, ArrayType, MapType

# Synapse Dedicated SQL Pool connector (in-box; no Databricks)
import com.microsoft.spark.sqlanalytics
from com.microsoft.spark.sqlanalytics.Constants import Constants

# Runtime utils: `mssparkutils` is injected in Synapse; newer runtimes expose
# `notebookutils`. Alias whichever exists so the rest of the notebook is portable.
try:
    mssparkutils  # noqa: F821  (injected global in Synapse)
except NameError:
    import notebookutils as mssparkutils

logger = logging.getLogger("brz_loader")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _h = logging.StreamHandler()
    _h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(_h)

RUN_ID = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:8]

# Normalize source_paths to a list
if isinstance(source_paths, str):
    SOURCE_PATHS = [p.strip() for p in source_paths.split(",") if p.strip()]
else:
    SOURCE_PATHS = [str(p).strip() for p in source_paths if str(p).strip()]

assert sql_endpoint,  "Parameter 'sql_endpoint' is required"
assert SOURCE_PATHS,  "Parameter 'source_paths' is required"
assert staging_path,  "Parameter 'staging_path' is required"

# Optional config overrides
CONFIG = {}
if config_path:
    try:
        CONFIG = json.loads(mssparkutils.fs.head(config_path, 10 * 1024 * 1024))
        logger.info("Loaded config overrides from " + config_path)
    except Exception as _e:
        logger.warning("Could not load config_path (" + str(config_path) + "): " + str(_e))

def _as_set(val):
    if not val:
        return set()
    if isinstance(val, str):
        return {x.strip() for x in val.split(",") if x.strip()}
    return {str(x).strip() for x in val if str(x).strip()}

# include/exclude come from the parameters AND (optionally) the config file; union of both
INCLUDE         = _as_set(include_tables) | _as_set(CONFIG.get("include"))
EXCLUDE         = _as_set(exclude_tables) | _as_set(CONFIG.get("exclude"))
CHANGE_TYPES    = CONFIG.get("change_types") or ["insert", "update_postimage", "update_preimage", "delete"]
DEFAULT_STRLEN  = int(CONFIG.get("string_len", string_len))
TABLE_OVERRIDES = CONFIG.get("table_overrides", {}) or {}

# Operational metadata columns appended to every bronze table (typed, not NVARCHAR)
META_COLS = ["_change_type", "_commit_version", "_commit_timestamp",
             "_load_run_id", "_loaded_utc", "_source_path"]

logger.info("run_id=" + RUN_ID)
logger.info("targets: " + database + "." + target_schema + "  ops: " + database + "." + ops_schema)

In [ ]:
# === Control-plane SQL over JDBC, authenticated with the workspace MI token ===
# Used for DDL, the idempotent transactional swap, watermark and audit writes.

def _jdbc_connection():
    token = mssparkutils.credentials.getToken(token_audience)
    jvm = spark._jvm
    url = ("jdbc:sqlserver://" + sql_endpoint + ":1433;database=" + database +
           ";encrypt=true;trustServerCertificate=false;"
           "hostNameInCertificate=*.sql.azuresynapse.net;loginTimeout=60")
    props = jvm.java.util.Properties()
    props.setProperty("accessToken", token)
    return jvm.java.sql.DriverManager.getConnection(url, props)

def sql_exec(statements):
    # Runs one or more statements in a single committed transaction (rolls back on error).
    if isinstance(statements, str):
        statements = [statements]
    conn = _jdbc_connection()
    try:
        conn.setAutoCommit(False)
        st = conn.createStatement()
        for s in statements:
            st.execute(s)
        conn.commit()
        st.close()
    except Exception:
        try:
            conn.rollback()
        except Exception:
            pass
        raise
    finally:
        conn.close()

def sql_scalar(query):
    conn = _jdbc_connection()
    try:
        st = conn.createStatement()
        rs = st.executeQuery(query)
        val = rs.getObject(1) if rs.next() else None
        rs.close(); st.close()
        return val
    finally:
        conn.close()

def sql_list(query):
    conn = _jdbc_connection()
    try:
        st = conn.createStatement()
        rs = st.executeQuery(query)
        out = []
        while rs.next():
            out.append(rs.getObject(1))
        rs.close(); st.close()
        return out
    finally:
        conn.close()

def q(val):
    # SQL string literal with single quotes escaped; None -> NULL
    if val is None:
        return "NULL"
    return "'" + str(val).replace("'", "''") + "'"

def ident(name):
    # Safely bracket-quote a SQL identifier (column/table), escaping any ']'
    return "[" + str(name).replace("]", "]]") + "]"

In [ ]:
# === Discover Delta tables (folders containing a _delta_log) ===

def sanitize(name):
    s = re.sub(r"[^0-9a-zA-Z_]", "_", name.strip().rstrip("/"))
    if s and s[0].isdigit():
        s = "t_" + s
    return s

def discover_delta_tables(roots):
    found = {}
    def walk(path):
        try:
            entries = mssparkutils.fs.ls(path)
        except Exception as e:
            logger.warning("Cannot list " + str(path) + ": " + str(e))
            return
        names = [e.name.rstrip("/") for e in entries]
        if "_delta_log" in names:
            base = path.rstrip("/")
            found[sanitize(base.split("/")[-1])] = base
            return  # a Delta table root; do not descend further
        for e in entries:
            if e.isDir:
                walk(e.path)
    for r in roots:
        walk(r)
    return found

In [ ]:
# === Read source Delta (full snapshot or CDF) and normalize to the target shape ===
from delta.tables import DeltaTable

def current_version(path):
    hist = DeltaTable.forPath(spark, path).history()
    return int(hist.agg(F.max("version").alias("v")).collect()[0]["v"])

def _cast_business(df, biz_cols):
    exprs = []
    for f in df.schema.fields:
        if f.name in biz_cols:
            if isinstance(f.dataType, (StructType, ArrayType, MapType)):
                exprs.append(F.to_json(F.col(f.name)).alias(f.name))   # complex -> JSON string
            else:
                exprs.append(F.col(f.name).cast("string").alias(f.name))
    return exprs

def _audit_cols(path):
    return [F.lit(RUN_ID).alias("_load_run_id"),
            F.current_timestamp().alias("_loaded_utc"),
            F.lit(path).alias("_source_path")]

def read_full(path, version):
    df = spark.read.format("delta").load(path)
    biz_cols = [f.name for f in df.schema.fields]
    sel = _cast_business(df, biz_cols)
    sel += [F.lit("insert").alias("_change_type"),
            F.lit(version).cast("long").alias("_commit_version"),
            F.lit(None).cast("timestamp").alias("_commit_timestamp")]
    sel += _audit_cols(path)
    return df.select(*sel), biz_cols

def read_cdf(path, start, end):
    df = (spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", start)
            .option("endingVersion", end)
            .load(path))
    cdf_meta = ("_change_type", "_commit_version", "_commit_timestamp")
    biz_cols = [f.name for f in df.schema.fields if f.name not in cdf_meta]
    if CHANGE_TYPES:
        df = df.filter(F.col("_change_type").isin(CHANGE_TYPES))
    sel = _cast_business(df, biz_cols)
    sel += [F.col("_change_type"),
            F.col("_commit_version").cast("long"),
            F.col("_commit_timestamp").cast("timestamp")]
    sel += _audit_cols(path)
    return df.select(*sel), biz_cols

In [ ]:
# === Target/staging DDL: ROUND_ROBIN + CLUSTERED COLUMNSTORE, all-NVARCHAR business cols ===

def table_exists(schema, name):
    return sql_scalar("SELECT OBJECT_ID('" + schema + "." + name + "')") is not None

def existing_columns(schema, name):
    rows = sql_list("SELECT column_name FROM information_schema.columns "
                    "WHERE table_schema = '" + schema + "' AND table_name = '" + name + "'")
    return [str(r) for r in rows]

def build_create_ddl(schema, name, biz_cols, n):
    cols = [ident(c) + " NVARCHAR(" + str(n) + ") NULL" for c in biz_cols]
    cols += ["[_change_type] NVARCHAR(64) NULL",
             "[_commit_version] BIGINT NULL",
             "[_commit_timestamp] DATETIME2(3) NULL",
             "[_load_run_id] NVARCHAR(64) NULL",
             "[_loaded_utc] DATETIME2(3) NULL",
             "[_source_path] NVARCHAR(2048) NULL"]
    return ("CREATE TABLE " + ident(schema) + "." + ident(name) + " (" + ", ".join(cols) +
            ") WITH (DISTRIBUTION = ROUND_ROBIN, CLUSTERED COLUMNSTORE INDEX)")

def ensure_table(schema, name, biz_cols, n):
    if not table_exists(schema, name):
        sql_exec(build_create_ddl(schema, name, biz_cols, n))
        logger.info("Created table " + schema + "." + name)
        return
    existing = set(existing_columns(schema, name))
    for c in [c for c in biz_cols if c not in existing]:
        sql_exec("ALTER TABLE " + ident(schema) + "." + ident(name) +
                 " ADD " + ident(c) + " NVARCHAR(" + str(n) + ") NULL")
        logger.info("Evolved " + schema + "." + name + ": added column " + c)

def ordered_columns(biz_cols):
    return biz_cols + META_COLS

def write_stage(df, stage_fqn):
    (df.write
       .option(Constants.SERVER, sql_endpoint)
       .option(Constants.TEMP_FOLDER, staging_path)
       .mode("append")
       .synapsesql(stage_fqn, Constants.INTERNAL))

In [ ]:
# === Audit logging to edw.operations.load_log (idempotent per run_id+table) ===

def _ts_lit(x):
    if x is None:
        return "NULL"
    return "'" + x.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3] + "'"

def _num(x):
    return "NULL" if x is None else str(x)

def write_log(info):
    dur = None
    if info.get("started_utc") and info.get("ended_utc"):
        dur = round((info["ended_utc"] - info["started_utc"]).total_seconds(), 3)
    stmts = [
        ("DELETE FROM " + ops_schema + ".load_log WHERE run_id = " + q(info["run_id"]) +
         " AND table_name = " + q(info["table_name"])),
        ("INSERT INTO " + ops_schema + ".load_log "
         "(run_id, table_name, source_path, phase, start_version, end_version, rows_loaded, "
         "status, error_message, started_utc, ended_utc, duration_sec) VALUES (" +
         q(info["run_id"]) + ", " + q(info["table_name"]) + ", " + q(info.get("source_path")) + ", " +
         q(info.get("phase")) + ", " + _num(info.get("start_version")) + ", " + _num(info.get("end_version")) + ", " +
         _num(info.get("rows_loaded") or 0) + ", " + q(info.get("status")) + ", " + q(info.get("error_message")) + ", " +
         _ts_lit(info.get("started_utc")) + ", " + _ts_lit(info.get("ended_utc")) + ", " +
         ("NULL" if dur is None else str(dur)) + ")"),
    ]
    try:
        sql_exec(stmts)
    except Exception as e:
        logger.warning("Could not write load_log for " + str(info.get("table_name")) + ": " + str(e))

In [ ]:
# === Per-table processing: idempotent, resilient, fully logged ===

def process_table(tbl, path):
    schema = target_schema
    stage  = tbl + "__stage"
    n      = int(TABLE_OVERRIDES.get(tbl, {}).get("string_len", DEFAULT_STRLEN))
    info = {"run_id": RUN_ID, "table_name": tbl, "source_path": path, "phase": None,
            "start_version": None, "end_version": None, "rows_loaded": 0,
            "status": None, "error_message": None,
            "started_utc": datetime.datetime.utcnow(), "ended_utc": None}
    try:
        V = current_version(path)
        last = sql_scalar("SELECT last_commit_version FROM " + ops_schema +
                          ".load_control WHERE table_name = " + q(tbl))
        last = None if last is None else int(last)
        target_present = table_exists(schema, tbl)

        do_initial = (last is None) or (not target_present)
        if do_initial:
            phase, start, end = "initial", 0, V
            df, biz_cols = read_full(path, V)
        else:
            if last >= V:
                info.update(phase="skipped", start_version=last, end_version=V, status="SKIPPED")
                logger.info("[" + tbl + "] up-to-date at version " + str(V) + " -> SKIPPED")
                return info
            phase, start, end = "incremental", last + 1, V
            try:
                df, biz_cols = read_cdf(path, start, end)
            except Exception as ce:
                logger.warning("[" + tbl + "] CDF read failed (" + str(ce) + "); full reload fallback")
                phase, start, do_initial = "initial", 0, True
                df, biz_cols = read_full(path, V)

        info.update(phase=phase, start_version=start, end_version=end)

        ensure_table(schema, tbl,   biz_cols, n)
        ensure_table(schema, stage, biz_cols, n)

        cols = ordered_columns(biz_cols)
        df = df.select(*cols)

        # Rebuild staging from scratch, then bulk-load this batch into it
        sql_exec("TRUNCATE TABLE " + ident(schema) + "." + ident(stage))
        cnt = df.count()
        info["rows_loaded"] = cnt
        if cnt > 0:
            write_stage(df, database + "." + schema + "." + stage)

        # Idempotent transactional swap + watermark advance (exactly-once per run)
        collist = ", ".join(ident(c) for c in cols)
        tx = []
        if do_initial:
            tx.append("TRUNCATE TABLE " + ident(schema) + "." + ident(tbl))
        else:
            tx.append("DELETE FROM " + ident(schema) + "." + ident(tbl) + " WHERE _commit_version BETWEEN " +
                      str(start) + " AND " + str(end))
        tx.append("INSERT INTO " + ident(schema) + "." + ident(tbl) + " (" + collist + ") SELECT " +
                  collist + " FROM " + ident(schema) + "." + ident(stage))
        tx.append("DELETE FROM " + ops_schema + ".load_control WHERE table_name = " + q(tbl))
        tx.append("INSERT INTO " + ops_schema + ".load_control "
                  "(table_name, source_path, last_commit_version, last_loaded_utc, last_run_id, last_status) "
                  "VALUES (" + q(tbl) + ", " + q(path) + ", " + str(V) +
                  ", SYSUTCDATETIME(), " + q(RUN_ID) + ", 'SUCCESS')")
        sql_exec(tx)

        info["status"] = "SUCCESS"
        logger.info("[" + tbl + "] " + phase + " load OK: " + str(cnt) +
                    " rows, versions " + str(start) + ".." + str(end))
    except Exception as e:
        info["status"] = "FAILED"
        info["error_message"] = (str(e) + " | " + traceback.format_exc())[:3900]
        logger.error("[" + tbl + "] FAILED: " + str(e))
    finally:
        info["ended_utc"] = datetime.datetime.utcnow()
        write_log(info)
    return info

In [ ]:
# === Orchestrate: discover, process (bounded parallelism), summarize ===

tables = discover_delta_tables(SOURCE_PATHS)
logger.info("Discovered " + str(len(tables)) + " Delta table(s): " + ", ".join(sorted(tables)))

if INCLUDE:
    missing = [t for t in INCLUDE if t not in tables]
    if missing:
        logger.warning("Requested include_tables not found under source_paths: " + ", ".join(sorted(missing)))
    tables = {k: v for k, v in tables.items() if k in INCLUDE}
if EXCLUDE:
    tables = {k: v for k, v in tables.items() if k not in EXCLUDE}

logger.info("Processing " + str(len(tables)) + " table(s): " + ", ".join(sorted(tables)))

results = []
if tables:
    with concurrent.futures.ThreadPoolExecutor(max_workers=max(1, int(max_workers))) as ex:
        futs = {ex.submit(process_table, t, p): t for t, p in tables.items()}
        for fut in concurrent.futures.as_completed(futs):
            results.append(fut.result())

summary = {
    "run_id":    RUN_ID,
    "total":     len(results),
    "succeeded": sum(1 for r in results if r["status"] == "SUCCESS"),
    "failed":    sum(1 for r in results if r["status"] == "FAILED"),
    "skipped":   sum(1 for r in results if r["status"] == "SKIPPED"),
    "rows":      sum(r.get("rows_loaded", 0) for r in results),
    "tables":    {r["table_name"]: r["status"] for r in results},
}
logger.info("RUN SUMMARY: " + json.dumps(summary))

write_log({"run_id": RUN_ID, "table_name": "__RUN_SUMMARY__", "source_path": None,
           "phase": "summary", "start_version": None, "end_version": None,
           "rows_loaded": summary["rows"],
           "status": ("FAILED" if summary["failed"] else "SUCCESS"),
           "error_message": None, "started_utc": None,
           "ended_utc": datetime.datetime.utcnow()})

failed_tables = [r["table_name"] for r in results if r["status"] == "FAILED"]
if fail_on_error and failed_tables:
    raise Exception("fail_on_error=True; failed tables: " + ", ".join(failed_tables))

mssparkutils.notebook.exit(json.dumps(summary))